# convert peak_locations.pkl to mat

In [ ]:
from scipy import io
import pickle

import nescafe23 as n23

In [ ]:
peak_locations_full = n23.open_peak_locations()
peak_locations_QC = peak_locations_full.copy()

In [ ]:
import numpy as np

# apply peak QC: keep flags 1 and 5, set everything else to NaN
# (a short/empty QC row is unreviewed, so the whole row is masked)
GOOD_QC = (1, 5)
qc_tables = {
    "": n23.PEAK_LOWER_QC,
    "upper_": n23.PEAK_UPPER_QC,
}

for prefix, qc_table in qc_tables.items():
    for var in ("times", "angles", "snrs"):
        for ray_id, values in peak_locations_QC[prefix + var].items():
            values = np.asarray(values, dtype=float)
            qc = np.asarray(qc_table.get(ray_id, []))
            good = np.zeros(values.shape, dtype=bool)
            if qc.shape == values.shape:
                good = np.isin(qc, GOOD_QC)
            peak_locations_QC[prefix + var][ray_id] = np.where(good, values, np.nan)

In [ ]:
def sanitize_keys(obj):
    """Recursively sanitize dict keys to be valid MATLAB field names."""
    if isinstance(obj, dict):
        new = {}
        for k, v in obj.items():
            # Replace +/- and leading digits; strip other bad chars
            new_k = str(k).replace('+', 'p').replace('-', 'm')
            if new_k[0].isdigit():
                new_k = 'f' + new_k
            new[new_k] = sanitize_keys(v)
        return new
    elif isinstance(obj, list):
        return [sanitize_keys(i) for i in obj]
    else:
        return obj

In [ ]:
io.savemat('peak_locations.mat', {'peak_locations_QC': sanitize_keys(peak_locations_QC), 'peak_locations_full': sanitize_keys(peak_locations_full)})

In [ ]:
import nescafe23 as n23

In [ ]:
dss = n23.open_receptions()

In [ ]:
dss[0].nominal_depth